In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 9.3 MB/s eta 0:00:00:00:0100:01
  Created wheel for gtfparse: filename=gtfparse-2.0.1-py3-none-any.whl size=15285 sha256=675b975e54bfd973f63a65204ae672294ebff5b8c892d8b2ce1dfe1050349142
  Stored in directory: /root/.cache/pip/wheels/91/da/d4/4168bc0aa594bfcda1ba95d81ea91552d506807d686ceb4e1e
Successfully built gtfparse
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 13.3 MB/s eta 0:00:0000:0100:01
Reason for being yanked: <none given>
  Attempting uninstall: polars
    Found existing installation: polars 0.18.4
    Uninstalling polars-0.18.4:
      Successfully uninstalled polars-0.18.4


In [2]:
!pip install pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.1/39.1 MB 12.0 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install anndata==0.8.0

In [4]:
from samalg import SAM
from Bio import SeqIO
from gtfparse import read_gtf
import pandas as pd
import pandas
import pyarrow
import pickle
import scanpy as sc
import numpy as np

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
#open scdata to see how many gene matches you have
dat = sc.read_h5ad('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_CJ_joined.h5ad')

In [19]:
dat.var_names

Index(['5S_rRNA-1', '5S_rRNA-10', '5S_rRNA-18', '5S_rRNA-20', '5S_rRNA-21',
       '5S_rRNA-23', '5S_rRNA-5', '5S_rRNA-8', '5S_rRNA-9', '7SK',
       ...
       'ZSWIM8', 'ZUP1', 'ZW10', 'ZWILCH', 'ZYG11A', 'ZYX', 'ZZEF1', 'ZZZ3',
       'prl', 'quPKCI'],
      dtype='object', length=20032)

In [20]:
input_file = open("../../cDNA_fasta/Coturnix_japonica.Coturnix_japonica_2.0.105.cdna.all.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    print(item)
    break

ID: ENSCJPT00005000007.1
Name: ENSCJPT00005000007.1
Description: ENSCJPT00005000007.1 cdna primary_assembly:Coturnix_japonica_2.0:MT:3965:4939:1 gene:ENSCJPG00005000007.1 gene_biotype:protein_coding transcript_biotype:protein_coding gene_symbol:ND1 description:NADH dehydrogenase subunit 1 [Source:NCBI gene;Acc:804668]
Number of features: 0
Seq('ATGACCCTATCAACCCTAACAAGTCTCATAATCATAACCCTATCCTATATAATT...TAA', SingleLetterAlphabet())


In [21]:
#pull longest gene
input_file = open("../../cDNA_fasta/Coturnix_japonica.Coturnix_japonica_2.0.105.cdna.all.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    #manually change gene_ID to match which field you would like to see in your BLAST table
    gene_ID = item.description.split('gene:')[1].split(' ')[0]
    if gene_ID in gene_dict.keys():
        if len(item.seq) > len(gene_dict[gene_ID].seq):
            gene_dict[gene_ID] = item
    else:
        gene_dict[gene_ID] = item
len(gene_dict)

15872

In [22]:
db_gene = read_gtf('../../cDNA_fasta/Coturnix_japonica.Coturnix_japonica_2.0.105.gtf', features = ['gene','transcript'])
df_gene = db_gene.to_pandas()

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_source', 'transcript_biotype', 'gene_name', 'transcript_name', 'tag']


In [23]:
#use if gene_id is not aligned between cdna and fasta ex: japanese quail
fti = []
for item in df_gene.index:
    fti.append(df_gene.loc[item,'gene_id'] + "." + df_gene.loc[item,'gene_version'])
df_gene['Full_gene_id'] = fti

In [24]:
df_gene

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_version,gene_source,gene_biotype,transcript_id,transcript_version,transcript_source,transcript_biotype,gene_name,transcript_name,tag,Full_gene_id
0,1,ensembl,gene,2704,5103,NaN,-,0,ENSCJPG00005000042,1,ensembl,protein_coding,,,,,,,,ENSCJPG00005000042.1
1,1,ensembl,transcript,2704,5103,NaN,-,0,ENSCJPG00005000042,1,ensembl,protein_coding,ENSCJPT00005000063,1,ensembl,protein_coding,,,,ENSCJPG00005000042.1
2,1,ensembl,gene,119229433,119232283,NaN,+,0,ENSCJPG00005000046,1,ensembl,protein_coding,,,,,UNC50,,,ENSCJPG00005000046.1
3,1,ensembl,transcript,119229433,119232283,NaN,+,0,ENSCJPG00005000046,1,ensembl,protein_coding,ENSCJPT00005000041,1,ensembl,protein_coding,UNC50,UNC50-201,,ENSCJPG00005000046.1
4,1,ensembl,gene,9060,22306,NaN,-,0,ENSCJPG00005000066,1,ensembl,pseudogene,,,,,,,,ENSCJPG00005000066.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59160,LSZS01009076.1,ensembl,transcript,21,1032,NaN,+,0,ENSCJPG00005000950,1,ensembl,lncRNA,ENSCJPT00005001517,1,ensembl,lncRNA,,,,ENSCJPG00005000950.1
59161,LSZS01009077.1,ensembl,gene,21,1031,NaN,-,0,ENSCJPG00005000952,1,ensembl,lncRNA,,,,,,,,ENSCJPG00005000952.1
59162,LSZS01009077.1,ensembl,transcript,21,1031,NaN,-,0,ENSCJPG00005000952,1,ensembl,lncRNA,ENSCJPT00005001520,1,ensembl,lncRNA,,,,ENSCJPG00005000952.1
59163,LSZS01009094.1,ensembl,gene,185,977,NaN,+,0,ENSCJPG00005006714,1,ensembl,protein_coding,,,,,,,,ENSCJPG00005006714.1


In [25]:
fin_gene_dict = {}
for item in gene_dict.keys():
    if len(df_gene.loc[df_gene.index[df_gene['Full_gene_id'] == item][0], 'gene_name']) > 0:
        fin_gene_dict[df_gene.loc[df_gene.index[df_gene['Full_gene_id'] == item][0], 'gene_name']] = gene_dict[item]
    else:
        fin_gene_dict[item.split('.')[0]] = gene_dict[item]

In [26]:
dat.var_names

Index(['5S_rRNA-1', '5S_rRNA-10', '5S_rRNA-18', '5S_rRNA-20', '5S_rRNA-21',
       '5S_rRNA-23', '5S_rRNA-5', '5S_rRNA-8', '5S_rRNA-9', '7SK',
       ...
       'ZSWIM8', 'ZUP1', 'ZW10', 'ZWILCH', 'ZYG11A', 'ZYX', 'ZZEF1', 'ZZZ3',
       'prl', 'quPKCI'],
      dtype='object', length=20032)

In [27]:
full_set_new = set(dat.var_names) & set(fin_gene_dict.keys())

In [28]:
len(full_set_new)

15178

In [29]:
for item in fin_gene_dict.keys():
    fin_gene_dict[item].id = item
    fin_gene_dict[item].name = item

with open("../../BLASTMAPPING/Coturnix_japonica.Coturnix_japonica_2.0.105.curated.08022026.fa", "w") as handle:
    SeqIO.write(fin_gene_dict.values(), handle, "fasta") 

In [19]:
input_file = open("../../BLASTMAPPING/Coturnix_japonica.Coturnix_japonica_2.0.105.curated.08012026.fa")
fin_list = []
for item in SeqIO.parse(input_file, "fasta"):
    fin_list.append(item.name)

In [20]:
len(set(dat.var_names) & set(fin_list))

15178